# MRKR contralateral TKA - train the ladder AND read the sealed test split, in ONE session

This notebook replaces the two-notebook flow, which lost its checkpoints when the runtime
disconnected. Colab wipes `/root` on teardown, and 35 checkpoints are about 5 GB, which is
too much to mirror to Drive reliably inside an idle timeout.

**The fix is to never need the transfer.** Train, then score the sealed split in the same
session, while the checkpoints are still on local disk. Only small artefacts leave.

### Order of operations
1-5. setup, smoke test, timing calibration
6. **Stage 1** (m0d_clinical, m1_klg, m4_fusion, m3_image)
7. **Stage 2** (m2_frontal, m4_frontal, r1_densenet_frontal)
8. **Checkpoint audit** - asserts 35 complete checkpoints exist before spending the test set
9. **THE SEALED READ** - `src/score_test.py`, inference only
10. **Package everything** - validation + test artefacts in one archive

### Non-negotiables
1. **Never downscale the 512x512 crop** (protocol section 13). If memory is tight, raise `--grad-accum`.
2. **The flags are load-bearing.** `grad_accum_steps`, `micro_batch_size` and `num_workers` are inside the training-contract hash. Pick them in section 5 and never change them.
3. **The test read happens once.** Section 9 requires `--confirm-sealed-read`. Any change to any model after it invalidates the estimate.

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM {p.total_memory/1024**3:.1f} GB")
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU, then rerun.")

## 2. Mount Drive and check the upload

Expected Drive layout. All of this is already there from previous runs:

```
MyDrive/mrkr/
    mrkr_colab_code.tar.gz      <- must contain src/score_test.py
    shards/
        train-00000.tar
        val-00000.tar
        labels.csv
    shards-test/
        test-00000.tar
        labels.csv
```

`ckpt/` is no longer used. Checkpoints stay on local disk for the life of the session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE   = pathlib.Path('/content/drive/MyDrive/mrkr')
BUNDLE  = DRIVE / 'mrkr_colab_code.tar.gz'
DSHARDS = DRIVE / 'shards'
DTEST   = DRIVE / 'shards-test'

for p in (BUNDLE,
          DSHARDS / 'train-00000.tar', DSHARDS / 'val-00000.tar', DSHARDS / 'labels.csv',
          DTEST / 'test-00000.tar', DTEST / 'labels.csv'):
    assert p.exists(), f"missing on Drive: {p}"
    print(f"ok  {str(p.relative_to(DRIVE)):32s} {p.stat().st_size/1048576:8.1f} MB")

## 3. Unpack code, stage BOTH shard sets on local SSD

`${HOME}` on Colab is `/root`, and `src/config.py` expands it, so `${HOME}/mrkr-shards`
resolves on its own. No config edit needed.

In [ ]:
import os, shutil, sys, tarfile, time

PROJ = pathlib.Path('/content/mrkr-project')
if PROJ.exists():
    shutil.rmtree(PROJ)
with tarfile.open(BUNDLE) as tf:
    tf.extractall('/content')
shutil.move('/content/project', str(PROJ))
assert (PROJ / 'src' / 'score_test.py').exists(), \
    "this bundle predates src/score_test.py - re-upload the rebuilt mrkr_colab_code.tar.gz"

t0 = time.time()
SH = pathlib.Path('/root/mrkr-shards'); SH.mkdir(parents=True, exist_ok=True)
for n in ('train-00000.tar', 'val-00000.tar', 'labels.csv'):
    if not (SH / n).exists():
        shutil.copy2(DSHARDS / n, SH / n)
ST = pathlib.Path('/root/mrkr-shards-test'); ST.mkdir(parents=True, exist_ok=True)
for n in ('test-00000.tar', 'labels.csv'):
    if not (ST / n).exists():
        shutil.copy2(DTEST / n, ST / n)
print(f"both shard sets staged in {time.time()-t0:.0f}s")

CK = pathlib.Path('/root/mrkr-ckpt'); CK.mkdir(parents=True, exist_ok=True)
os.chdir(PROJ)
sys.path.insert(0, str(PROJ))
print(pathlib.Path.cwd())

## 4. Dependencies

Colab ships a CUDA-built torch. Do **not** `pip install -r requirements-training.txt`: its
`torch==2.13.0` pin would replace it with a build that may not match the driver.

In [ ]:
!pip -q install timm lifelines duckdb 2>&1 | tail -2

import importlib
for m in ("timm", "lifelines", "duckdb", "patsy", "statsmodels", "pyarrow", "sklearn", "yaml", "PIL"):
    try:
        mod = importlib.import_module(m)
        print(f"  {m:14s} {getattr(mod, '__version__', 'ok')}")
    except ImportError as e:
        print(f"  {m:14s} MISSING -> {e}")

## 5. Smoke test, then set the flags

| GPU | VRAM | start with |
|---|---|---|
| T4 | 16 GB | `--grad-accum 8` |
| L4 | 24 GB | `--grad-accum 4` |
| A100 | 40 GB | `--grad-accum 4` |

Use the SAME values you used before (`GRAD_ACCUM = 4`, `NUM_WORKERS = 2`) so the contract
hash comes out as `4b862b5ecb947314` and the models match the ones already characterised.

In [ ]:
!python -m src.train_model --smoke

In [ ]:
GRAD_ACCUM  = 4
NUM_WORKERS = 2

!python -m src.train_model --time-steps 20 --grad-accum {GRAD_ACCUM} --num-workers {NUM_WORKERS}

## 6. Stage 1 - the manuscript-critical arms

`m0d_clinical`, `m1_klg`, `m4_fusion` (primary), `m3_image`. Roughly 45 minutes.

**Keep this tab focused and active.** The runtime disconnected on idle last time; that is
what cost us the checkpoints.

In [ ]:
!python -m src.train_model --stage stage1 --grad-accum {GRAD_ACCUM} --num-workers {NUM_WORKERS}

## 7. Stage 2 - protocol completeness

`m2_frontal`, `m4_frontal` (section 24 views), `r1_densenet_frontal` (section 25
robustness, now ConvNeXt-Tiny since DenseNet121 became the primary backbone).

In [ ]:
!python -m src.train_model --stage stage2 --grad-accum {GRAD_ACCUM} --num-workers {NUM_WORKERS}

## 8. Checkpoint audit - run this BEFORE the sealed read

The test split is spent once. Confirm every model is actually present and complete first,
so a half-trained ladder cannot consume it.

In [ ]:
import json, torch

files = sorted(CK.glob('*.pt'))
total = sum(f.stat().st_size for f in files) / 1073741824
print(f"checkpoints on local disk: {len(files)}  ({total:.1f} GB)")

ta = json.loads(pathlib.Path('derived-data/cohort/train_arms.json').read_text())
arms = ta['arms']
print(f"arms in hand-over index: {len(arms)}  | contract {ta['training_contract_hash']}")

bad = []
for a, s in arms.items():
    if not s.get('complete'):
        bad.append(f"{a}: not complete")
    for seed in s['seeds']:
        f = CK / f"{a}_seed{seed}.pt"
        if not f.exists():
            bad.append(f"{a} seed {seed}: checkpoint missing")
print(f"expected {sum(len(s['seeds']) for s in arms.values())} checkpoints")
assert not bad, "REFUSING to read the test split:\n  " + "\n  ".join(bad)
print("\nall arms complete and every checkpoint present - safe to proceed")

## 9. THE SEALED READ

Inference only. `score_test.py` asserts the frozen contract hash matches the live config
before it touches a test row, refuses any arm not marked complete, and applies the
**validation-fitted** recalibration as-is. It cannot train, refit, or re-select an epoch.

Run the refusal check first: it should stop with a message about `--confirm-sealed-read`.

In [ ]:
# Expected to REFUSE. If it does anything else, stop.
!python -m src.score_test --shard-dir /root/mrkr-shards-test

In [ ]:
!python -m src.score_test --shard-dir /root/mrkr-shards-test --confirm-sealed-read

## 10. Package everything and copy to Drive

Validation and test artefacts from the SAME models, in one archive. Small enough that the
copy cannot fail the way 5 GB of checkpoints did.

The npz files carry per-patient hazards keyed by `empi_anon`, so they are patient-level:
they belong in `derived-data/`, which is git-ignored, and must never be committed.

In [ ]:
OUTT = pathlib.Path('/content/mrkr_all_results.tar.gz')
coh = pathlib.Path('derived-data/cohort')
want = (sorted(coh.glob('val_hazards_*.npz')) + sorted(coh.glob('test_hazards_*.npz'))
        + [coh / 'train_arms.json', coh / 'test_scoring.json',
           pathlib.Path('outputs/tables/train_history.csv'),
           pathlib.Path('outputs/tables/seed_variability.csv')])
missing = [str(p) for p in want if not p.exists()]
assert not missing, f"not produced: {missing}"

with tarfile.open(OUTT, 'w:gz') as tf:
    for p in want:
        tf.add(p, arcname=str(p))
        print(f"  {str(p):56s} {p.stat().st_size/1024:8.1f} KB")

shutil.copy2(OUTT, DRIVE / OUTT.name)
print(f"\nwrote {OUTT} ({OUTT.stat().st_size/1048576:.1f} MB) and copied to Drive")
print(f"val arms: {len(sorted(coh.glob('val_hazards_*.npz')))} | "
      f"test arms: {len(sorted(coh.glob('test_hazards_*.npz')))}")

In [ ]:
import pandas as pd
print(pd.read_csv('outputs/tables/seed_variability.csv').to_string(index=False))